In [1]:
import sys
sys.path.insert(0, "..")

import glob


from hnrs.models.lenet import load_and_predict

ModuleNotFoundError: No module named 'keras'

In [ ]:
import numpy as np
from PIL import Image, ImageFilter


def _stroke_width(mask):
    """Estimate stroke width as 2 * area / edge pixels (a strip of width W gives W)."""
    area = mask.sum()
    padded = np.pad(mask, 1)
    interior = (padded[1:-1, 1:-1] & padded[:-2, 1:-1] & padded[2:, 1:-1]
                & padded[1:-1, :-2] & padded[1:-1, 2:])
    edge = area - interior.sum()
    return 2 * area / max(edge, 1)


def _otsu_threshold(arr):
    """Otsu's method: the brightness cutoff that best separates ink from background."""
    hist = np.bincount(arr.astype(np.uint8).ravel(), minlength=256).astype(np.float64)
    p = hist / hist.sum()
    omega = np.cumsum(p)
    mu = np.cumsum(p * np.arange(256))
    between = (mu[-1] * omega - mu) ** 2 / (omega * (1 - omega) + 1e-12)
    return int(np.argmax(between))


def preprocess_image(source, box=20, size=28):
    """Convert any digit image into the MNIST format the models were trained on.

    Steps: greyscale, fix orientation and lighting, separate ink from background,
    crop to the digit, thicken thin strokes, fit into 20x20, centre on 28x28.

    source : file path, PIL Image, or numpy array (photo, drawing, or segmentation crop)
    returns: (28, 28) float32 array, values 0 to 1, white digit on black,
             scaled to fit a 20x20 box and centred by centre of mass, as in MNIST
    """
    # 1. Load. Flatten any transparency onto white, then convert to greyscale.
    if isinstance(source, np.ndarray):
        img = Image.fromarray(source)
    elif isinstance(source, Image.Image):
        img = source
    else:
        img = Image.open(source)
    if img.mode in ("RGBA", "LA", "P"):
        img = img.convert("RGBA")
        img = Image.alpha_composite(Image.new("RGBA", img.size, (255, 255, 255, 255)), img)
    arr = np.array(img.convert("L"), dtype=np.float32)

    # 2. Shrink very large images first. The output is 28 px, so detail beyond
    #    this is wasted, and everything after runs much faster.
    h, w = arr.shape
    if max(h, w) > 512:
        s = 512 / max(h, w)
        arr = np.array(Image.fromarray(arr.astype(np.uint8)).resize(
            (max(1, round(w * s)), max(1, round(h * s))), Image.BOX), dtype=np.float32)

    # 3. Work out which way round the image is and even out the lighting.
    if max(arr.shape) >= 100:
        # Larger images (photos, drawings): a heavy blur estimates the background
        # brightness at every point. The ink is whatever stands out most from its
        # local background, whether darker or lighter, which copes with shadows
        # and dark corners that would fool a simple check of the border.
        blur = Image.fromarray(arr.astype(np.uint8)).filter(
            ImageFilter.GaussianBlur(radius=max(arr.shape) / 8))
        diff = arr - np.array(blur, dtype=np.float32)
        if -np.percentile(diff, 0.5) > np.percentile(diff, 99.5):
            diff = -diff  # dark ink on light paper
        arr = np.clip(diff, 0, None)
    else:
        # Small images (MNIST digits, segmentation crops) are already clean, so
        # the border, which is mostly background, is enough to decide.
        border = np.concatenate([arr[0], arr[-1], arr[:, 0], arr[:, -1]])
        if np.median(border) > 127:
            arr = 255 - arr

    # 4. Separate ink from background with Otsu's method, which picks the cutoff
    #    from the image's own brightness histogram.
    if arr.max() - arr.min() < 30:
        raise ValueError("No digit found in image")
    arr = arr / arr.max() * 255
    arr[arr <= _otsu_threshold(arr)] = 0

    # 5. Crop to the ink.
    rows = np.where(arr.any(axis=1))[0]
    cols = np.where(arr.any(axis=0))[0]
    arr = arr[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]

    # 6. Shrink to an intermediate size (longest side 100 px), keeping aspect ratio.
    h, w = arr.shape
    s = 100 / max(h, w)
    mid = Image.fromarray((arr / arr.max() * 255).astype(np.uint8))
    mid = mid.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BOX)

    # 7. Thicken thin strokes. MNIST strokes are roughly 2 px in the 20 px box,
    #    which is about 10 px at this intermediate size.
    width = _stroke_width(np.array(mid) > 127)
    extra = int(round(10 - width))
    if extra >= 2:
        mid = mid.filter(ImageFilter.MaxFilter(extra // 2 * 2 + 1))

    # 8. Resize so the longest side is 20 px.
    mw, mh = mid.size
    s = box / max(mw, mh)
    new_w, new_h = max(1, round(mw * s)), max(1, round(mh * s))
    digit = np.array(mid.resize((new_w, new_h), Image.BOX), dtype=np.float32)

    # 9. Place on a 28x28 canvas with the centre of mass in the middle.
    total = digit.sum()
    cy = (digit.sum(axis=1) * np.arange(new_h)).sum() / total
    cx = (digit.sum(axis=0) * np.arange(new_w)).sum() / total
    top = int(np.clip(round(size / 2 - cy), 0, size - new_h))
    left = int(np.clip(round(size / 2 - cx), 0, size - new_w))
    canvas = np.zeros((size, size), dtype=np.float32)
    canvas[top:top + new_h, left:left + new_w] = digit

    # 10. Scale to 0 to 1.
    return canvas / canvas.max()

In [4]:
path = sorted(glob.glob("../data/handwritten/*.png"))[0]
mnist = sorted(glob.glob("../data/imgs/9/*.png"))[0]

image = np.array(Image.open(path)).astype(np.float32) / 255.0
image = np.expand_dims(image, axis=-1)

mnist_image = np.array(Image.open(mnist)).astype(np.float32) / 255.0
mnist_image = np.expand_dims(mnist_image, axis=-1)


In [ ]:
print(path)
print("Sketch.io 9: " + str(image.shape) + ", " + str(image.dtype) + ", " + str(image.min()) + ", " + str(image.max()))

print(mnist)
print("MNIST 9: " + str(mnist_image.shape) + ", " + str(mnist_image.dtype) + ", " + str(mnist_image.min()) + ", " + str(mnist_image.max()))


../data/handwritten/nine.png
Sketch.io 9: (2615, 2440, 4, 1), float32, 0.0, 1.0
../data/imgs/9/index_00004.png
MNIST 9: (28, 28, 1), float32, 0.0, 1.0


In [ ]:
image = preprocess_image(path)
print(image.shape, image.dtype, image.min(), image.max(), round(float(image.mean()), 3))

image = np.expand_dims(image, axis=-1)
label, confidence = load_and_predict(image)
print(f"Predicted label: {label}, confidence: {confidence:.4f}")

In [ ]:
print(path)
print("Sketch.io 9: " + str(image.shape) + ", " + str(image.dtype) + ", " + str(image.min()) + ", " + str(image.max()))

print(mnist)
print("MNIST 9: " + str(mnist_image.shape) + ", " + str(mnist_image.dtype) + ", " + str(mnist_image.min()) + ", " + str(mnist_image.max()))


In [8]:
label, confidence = load_and_predict(mnist_image)
print(f"Predicted label: {label}, confidence: {confidence:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
Predicted label: 9, confidence: 0.9998


In [15]:
label, confidence = load_and_predict(image)
print(f"Predicted label: {label}, confidence: {confidence:.4f}")

ValueError: Exception encountered when calling Sequential.call().

[1mCannot take the length of shape with unknown rank.[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=<unknown>, dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>